# MWFG Engagement Intelligence Agent — Capstone

---

## 🏗️ Build | 🚢 Ship | 📤 Share

### 🏗️ Build
- Wire together the harness, RAG pipeline, memory system, skills, and MCP server from the earlier notebooks
- A single agent with 7 tools spanning retrieval, memory, structured data, and arithmetic
- A Streamlit UI that shows which tools fired on each turn

### 🚢 Ship
A production-quality consulting assistant that remembers facts across sessions, retrieves documents by meaning, and queries live engagement data — all through one unified interface.

### 📤 Share
Run the agent against 10 benchmark questions and compute a tool-accuracy score using the LLM-as-judge pattern. Did the agent pick the right tool for each question type?

---

Everything you've built across Day 1 and Day 2 comes together here:

| Component | Source |
|-----------|-------|
| Agent harness | Agent Harness notebook |
| RAG retrieval | RAG pipeline notebook |
| Persistent memory | Agent Memory notebook |
| Skills library | MCPs & Skills notebook |
| MCP server tools | MCPs & Skills notebook |
| Streamlit UI | Accenture Engagement Agent |
| Evaluation | LLM-as-Judge notebook |

**Estimated time:** 60–75 minutes

---

In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Make agent/ importable
sys.path.insert(0, str(Path(".")))

load_dotenv(Path("..") / ".env")
assert os.environ.get("OPENAI_API_KEY") or os.environ.get("ANTHROPIC_API_KEY")

MODEL = "gpt-4o-mini"
DATA_DIR = Path(".") / "data"
MEMORY_FILE = DATA_DIR / "memory.json"
print("✅ Ready")


# --- local model routing (reads LLM_MODEL / LLM_API_BASE from .env) ---
import functools as _functools
import litellm as _litellm

_LOCAL_MODEL      = os.getenv('LLM_MODEL',      'gpt-4o-mini')
_LOCAL_API_BASE   = os.getenv('LLM_API_BASE')   or None
_LOCAL_EMBED      = os.getenv('EMBED_MODEL',    'text-embedding-3-small')
_LOCAL_EMBED_BASE = os.getenv('EMBED_API_BASE') or None

_OPENAI_MODELS = {'gpt-4o-mini', 'gpt-4o', 'gpt-4-turbo', 'gpt-3.5-turbo',
                   'openai/gpt-4o-mini', 'openai/gpt-4o'}
_OPENAI_EMBEDS = {'text-embedding-3-small', 'text-embedding-3-large',
                   'text-embedding-ada-002'}

MODEL      = _LOCAL_MODEL
EMBED_MODEL = _LOCAL_EMBED

if _LOCAL_API_BASE:
    _orig_comp = _litellm.completion
    @_functools.wraps(_orig_comp)
    def _comp(*a, **kw):
        if kw.get('model') in _OPENAI_MODELS:
            kw['model'] = _LOCAL_MODEL
        kw.setdefault('api_base', _LOCAL_API_BASE)
        kw.setdefault('max_tokens', 2048)
        return _orig_comp(*a, **kw)
    _litellm.completion = _comp

    _orig_emb = _litellm.embedding
    @_functools.wraps(_orig_emb)
    def _emb(*a, **kw):
        if kw.get('model') in _OPENAI_EMBEDS:
            kw['model'] = _LOCAL_EMBED
        kw.setdefault('api_base', _LOCAL_EMBED_BASE or _LOCAL_API_BASE)
        return _orig_emb(*a, **kw)
    _litellm.embedding = _emb

print(f'Model: {MODEL}  API base: {_LOCAL_API_BASE or "default"}')
# --- end local routing ---


✅ Ready


Model: openai/nvidia-nemotron-3-super-120b-a12b  API base: http://192.168.1.79:8080/v1


---

## Task 1 — Assembling the parts

Each import below comes from a module you've already built. If any of them fail, go back to that module and make sure it works standalone.

In [2]:
from agent.harness import Agent, Tool
from agent.memory import Memory
from agent.rag import RAGIndex
from agent.skills import load_skills

print("All imports OK")

# Index the engagement document corpus
print("Indexing documents...")
rag = RAGIndex(DATA_DIR)
print(f"  → indexed documents from {DATA_DIR}")

# Load memory
memory = Memory(MEMORY_FILE)
print(f"  → memory loaded ({len(memory.all_facts())} long-term facts)")

# Load skills
skills = load_skills()
print(f"  → {len(skills)} skills loaded: {[s.name for s in skills]}")

All imports OK
Indexing documents...


  → indexed documents from data
  → memory loaded (0 long-term facts)
  → 2 skills loaded: ['search_engagement', 'calculate']


---

## Task 2 — Building the full tool set

In [3]:
# Import MCP server functions
sys.path.insert(0, str(Path("../07_MCPs_and_Skills")))
from mcp_server import get_workstream_status, get_risks

# Build the complete tool set
tools = skills + [
    # RAG retrieval
    Tool(
        name="rag_search",
        description="Search the engagement document corpus for relevant information. "
                    "Use this for detailed questions about the MWFG engagement.",
        parameters={"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        handler=rag.search,
    ),
    # Memory tools
    Tool(
        name="remember",
        description="Store a fact in long-term memory for future sessions.",
        parameters={
            "type": "object",
            "properties": {"key": {"type": "string"}, "value": {"type": "string"}},
            "required": ["key", "value"],
        },
        handler=lambda key, value: memory.set(key, value),
    ),
    Tool(
        name="recall",
        description="Search semantic memory for facts related to a query.",
        parameters={"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        handler=lambda query: "\n".join(memory.recall(query)) or "Nothing found in memory.",
    ),
    # MCP tools
    Tool(
        name="get_workstream_status",
        description="Get live workstream status from the MWFG engagement system (ws1, ws2, ws3).",
        parameters={"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]},
        handler=get_workstream_status,
    ),
    Tool(
        name="get_risks",
        description="Get risks by severity level from the MWFG engagement system (High, Medium, Low).",
        parameters={
            "type": "object",
            "properties": {"severity": {"type": "string", "enum": ["High", "Medium", "Low"]}},
            "required": ["severity"],
        },
        handler=get_risks,
    ),
]

print(f"Total tools: {len(tools)}")
for t in tools:
    print(f"  - {t.name}")

Total tools: 7
  - search_engagement
  - calculate
  - rag_search
  - remember
  - recall
  - get_workstream_status
  - get_risks


---

## Task 3 — Building the full agent

In [4]:
BASE_SYSTEM = """You are the MWFG Engagement Intelligence Agent — an AI assistant for the
Accenture team working on the MidWest Financial Group AI transformation engagement.

Tool usage guide:
- Use `rag_search` for detailed questions about documents, financials, or strategy
- Use `get_workstream_status` or `get_risks` for structured engagement data
- Use `search_engagement` for quick factsheet lookups
- Use `calculate` for numerical analysis
- Use `remember` to store important facts the user tells you
- Use `recall` to retrieve past analysis or notes

Always be concise, professional, and cite your sources."""


def build_agent(user_question: str = "") -> Agent:
    """Build the agent with memory context injected into the system prompt."""
    mem_context = memory.context_for(user_question)
    system = BASE_SYSTEM + (f"\n\n{mem_context}" if mem_context else "")
    return Agent(system=system, tools=tools, model=MODEL)


agent = build_agent()
print("Agent ready with", len(tools), "tools")

Agent ready with 7 tools


---

## Task 4 — Running end-to-end

In [5]:
# Test with a question that requires RAG
print("=== RAG + Skills ===")
for token in agent.stream("Based on the AI strategy brief, what is the build vs buy decision for the RAG pipeline?"):
    print(token, end="", flush=True)
print()

00:11:38 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider = openai


=== RAG + Skills ===


[05/17/26 00:11:38] INFO                                                                              ]8;id=1596519;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py\utils.py]8;;\:]8;id=1596520;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py#4004\4004]8;;\
                             LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider               
                             = openai                                                                              


[tool: search_engagement] → **Objective:** Deflect 30% of routine inquiries to AI agent, reducing call center costs by $2.8M/year  
**Approach:** RA...


00:11:50 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider = openai


...

 more

 advantageous

 than

 purchasing

.

In [6]:
# Test with a question that uses the MCP server
print("=== MCP Tools ===")
for token in agent.stream("What are all our High-severity risks and what's the status of WS2?"):
    print(token, end="", flush=True)
print()

00:12:14 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider = openai


=== MCP Tools ===


[05/17/26 00:12:14] INFO                                                                              ]8;id=1596544;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py\utils.py]8;;\:]8;id=1596545;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py#4004\4004]8;;\
                             LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider               
                             = openai                                                                              


[tool: get_risks] → **High severity risks:**
- [R1] PII handling approval delays WS2 launch (Owner: MWFG Legal + Accenture PMO, Status: Acti...


00:12:17 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider = openai


...

get

_work

stream

_status

】

In [7]:
# Test memory persistence
print("=== Memory ===")
agent.chat("My name is James and I'm the CDO. Please remember that I prefer executive-level summaries.") 

# Check that it was stored
print("Long-term memory after:", memory.all_facts())

00:12:29 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider = openai


=== Memory ===


[05/17/26 00:12:29] INFO                                                                              ]8;id=1596562;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py\utils.py]8;;\:]8;id=1596563;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py#4004\4004]8;;\
                             LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider               
                             = openai                                                                              

00:12:32 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider = openai


[05/17/26 00:12:32] INFO                                                                              ]8;id=1596568;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py\utils.py]8;;\:]8;id=1596569;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py#4004\4004]8;;\
                             LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider               
                             = openai                                                                              

Long-term memory after: {'james_cdo_summary_style': 'Prefers executive-level summaries.'}


In [8]:
# Simulate a restart — build a fresh agent and verify memory is recalled
print("=== Memory after restart ===")
fresh_agent = build_agent(user_question="Who am I?")
print(fresh_agent.chat("Do you know who I am?"))

00:12:37 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider = openai


=== Memory after restart ===


[05/17/26 00:12:37] INFO                                                                              ]8;id=1596574;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py\utils.py]8;;\:]8;id=1596575;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py#4004\4004]8;;\
                             LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider               
                             = openai                                                                              

00:12:42 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider = openai


[05/17/26 00:12:42] INFO                                                                              ]8;id=1596580;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py\utils.py]8;;\:]8;id=1596581;file:///home/imjonezz/Desktop/accenture_evals/.venv/lib/python3.11/site-packages/litellm/utils.py#4004\4004]8;;\
                             LiteLLM completion() model= nvidia-nemotron-3-super-120b-a12b; provider               
                             = openai                                                                              




[tool: recall] → Nothing found in memory.


No, I currently don't know who you are—I haven't been provided with your name or personal details yet. If there's something from a previous conversation that would help me identify you (like an email address, role on the MWFG engagement team, or other context), please share it and I'll be happy to remember for future reference.

How can I assist you today?


---

## Task 5 — The Streamlit app

Run the app and test it in a browser:

```bash
pixi run capstone
```

The app is in `app.py`. Open it to see how the UI wires everything together.

---

## 🎯 Activity — Eval your agent

Use the LLM-as-Judge evaluation pattern from the LLM-as-Judge notebook to formally evaluate the capstone agent. Design a judge for:

**"Did the agent use the right tool for the question?"**

For 10 questions, the correct tool is:
- Factual from factsheet → `search_engagement` or `rag_search`
- Structured engagement data → `get_workstream_status` or `get_risks`
- Numerical calculation → `calculate`
- Memory retrieval → `recall`

In [9]:
# Test questions for eval
test_questions = [
    "What is the total contract value?",                          # search_engagement
    "What percentage of the budget has been spent?",              # calculate
    "What is the current status of WS1?",                        # get_workstream_status
    "Who owns the Genesys integration risk?",                    # get_risks
    "What does the AI strategy brief say about model risk?",     # rag_search
    "What is 2.575 / 4.2 as a percentage?",                     # calculate
    "What are all High-severity risks?",                         # get_risks
    "What is the WS3 outcome?",                                  # get_workstream_status
    "According to the FAQ, what happens if model accuracy drops?", # rag_search
    "Who is the client sponsor?",                                # search_engagement
]

# Your eval implementation here
# 1. Run each question through the agent and capture which tools were called
# 2. Compare to expected tool list above
# 3. Compute "tool accuracy" score
# 4. For failures: write an LLM-as-judge prompt to explain why the wrong tool was used


---

## What We Just Built vs. What's In Production

| What we built | Production equivalent |
|---------------|----------------------|
| `Agent` class from the harness notebook | An orchestration framework (LangGraph, CrewAI, AutoGen) managing state machines and retries |
| `RAGIndex` over a local `data/` directory | A managed vector store (Pinecone, Weaviate) with continuous ingestion from SharePoint, Confluence, or S3 |
| `Memory` with JSON + ChromaDB | A memory service behind an API (user profiles in Postgres, semantic memory in pgvector) |
| `load_skills()` from a local directory | A tool registry with versioned, signed tool packages deployed independently of the agent |
| `mcp_server.py` imported directly | An MCP server running as a microservice, secured with OAuth and auto-scaled via Kubernetes |
| Streamlit app | A React/Next.js frontend with SSE streaming, session management, and audit logging |
| LLM-as-Judge eval in a notebook | A CI/CD eval pipeline that gates every deployment on a regression suite across 100+ test cases |

---

## 🚀 Advanced Build

**Multi-agent handoff:** The capstone agent handles all questions itself. In production, a coordinator routes requests to specialist subagents — a Risk Analyst agent, a Budget Analyst agent, a Document Retrieval agent — and synthesizes their outputs. Extend `agent/harness.py` with a `delegate(agent, question)` method and build a two-level coordinator that handles the MWFG engagement use case.

**Streaming eval:** The current eval runs questions synchronously and checks which tools were called. Add a streaming harness that captures tool calls in real time (via the `stream()` generator), measures time-to-first-token and time-to-first-tool-call, and reports both accuracy and latency in one eval table.